<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/06_pandas_intro/06_2_DataFrame_Basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06.2 — DataFrame Basics

## Learning Objectives

By the end of this notebook you should be able to:

- Describe a DataFrame as a table whose columns are Series
- Load a CSV file from disk or a URL with `pd.read_csv()`
- Orient yourself on a new dataset using `.head()`, `.shape`, `.dtypes`, `.info()`, and `.describe()`
- Select one column (returns a Series) and multiple columns (returns a DataFrame)
- Add a new computed column and drop unwanted columns

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_titanic.csv"
df = pd.read_csv(url)
df.columns = df.columns.str.lower()
df = df.rename(columns={
    'siblings/spouses aboard': 'sibsp',
    'parents/children aboard': 'parch'
})
df.head()

## 1. What You're Looking At

You just loaded a **DataFrame** — a two-dimensional table, like a spreadsheet. Each row is one passenger; each column is one piece of information about that passenger.

In the previous notebook you worked with a single column (a **Series**). A DataFrame is simply a collection of Series, all sharing the same row index. Every column you see above — `survived`, `pclass`, `name`, `sex`, `age`, `sibsp`, `parch`, `fare` — is its own Series. They happen to be organized side by side into a table.

Before you can analyze anything, you need to orient yourself: How many passengers are recorded? What does each column contain? Are the data types sensible? Are there gaps? The next few cells answer those questions.

## 2. Understanding the Shape and Structure

The very first thing to check: how big is this dataset?

In [ ]:
rows, cols = df.shape
print(f"This dataset has {rows} passengers and {cols} columns.")

Next, look at the first and last few rows. This helps you spot obvious problems — columns in the wrong order, values that look garbled, rows that shouldn't be there.

In [ ]:
df.head()   # first 5 rows (default)

In [ ]:
df.tail()   # last 5 rows — sometimes the end of a file has junk rows

## 3. Data Types — Are They What You Expect?

pandas assigns a **data type** to each column when it reads the file. It mostly guesses correctly, but not always. Check with `.dtypes`.

In [ ]:
df.dtypes

A few things to notice:
- `age` and `fare` are `float64` — numeric, as expected.
- `survived` and `pclass` are `int64`. They look like numbers, but they're really *categories* (0/1 for survived; 1/2/3 for class). pandas doesn't know that yet.
- `name` and `sex` are `object` — pandas' way of saying "string."

Knowing the types matters because operations behave differently on numbers vs strings. We'll fix the categorical columns in the Data Cleaning notebook.

## 4. A Single Call That Tells You Almost Everything

`.info()` combines shape, column names, types, and missing-value counts into one printout. Make it your standard first call on any new dataset.

In [ ]:
df.info()

The "Non-Null Count" column is especially important. If any column shows fewer than 887 non-null values, that column has missing data. This dataset is complete — all 887 rows have values in all 8 columns — which is unusually clean.

## 5. Quick Statistics with `.describe()`

Before asking specific questions, it's useful to get a feel for the *distribution* of the numeric columns. `.describe()` gives you count, mean, standard deviation, and the five-number summary (min, quartiles, max) in one table.

In [ ]:
df.describe()

Take a moment to look at the numbers:

- The average `age` is about 29. The youngest passenger was under 1 year old.
- `fare` ranges from £0 to over £500 — a massive spread. The median (50%) is only £14, so most passengers paid relatively little while a few paid enormous amounts.
- `survived` has a mean of 0.38, meaning 38% of passengers survived.
- `pclass` has a mean of 2.3 — there were more third-class passengers than first.

These summary numbers raise questions. *Why did some passengers pay £0? Who paid £500? What explains the 38% survival rate?* The rest of this notebook series is about answering questions like these.

## 6. Selecting Columns

You've seen the full table. Now suppose you want to focus on just a few columns — maybe you're investigating the relationship between age, sex, and survival and don't need `sibsp`, `parch`, or `fare` cluttering your view.

### Single column → Series

Use `df["column_name"]`. The result is a `Series` — exactly what you worked with in notebook 06.1.

In [ ]:
ages = df["age"]
print(type(ages))      # pandas Series
ages.head()

### Multiple columns → DataFrame

Pass a **list** of column names inside the brackets. The result is a smaller DataFrame.

The double brackets are intentional: the outer `[ ]` is the selection operator; the inner `[ ]` is a Python list of names. This is a common point of confusion early on.

In [ ]:
core = df[["name", "sex", "age", "pclass", "survived"]]
core.head()

Think of this as "projecting" the table down to only the columns you care about. You're not modifying the original `df` — you're creating a new view with fewer columns.

## 7. Adding a New Column

Sometimes the information you need isn't directly in the dataset — you have to compute it. You can add a new column by assigning an expression to a new column name.

For example: `sibsp` counts siblings and spouses; `parch` counts parents and children. Neither tells you directly whether a passenger was traveling alone. Let's create a column that does.

In [ ]:
# A passenger had family aboard if sibsp OR parch is greater than 0
df["has_family"] = (df["sibsp"] > 0) | (df["parch"] > 0)

df[["name", "sibsp", "parch", "has_family"]].head(10)

In [ ]:
# How many passengers traveled with family vs alone?
print(df["has_family"].value_counts())
print()
print(f"Alone: {(~df['has_family']).sum()}")
print(f"With family: {df['has_family'].sum()}")

This kind of **feature engineering** — deriving new columns from existing ones — is something you'll do constantly. It lets you represent ideas that the raw data doesn't capture directly.

## 8. Dropping Unwanted Columns

Once you know which columns you need, it's often cleaner to drop the ones you don't. `.drop()` returns a **new** DataFrame — it does not modify the original.

- Drop columns: `df.drop(columns=["col1", "col2"])`
- Drop rows: `df.drop(index=[0, 5, 10])` (by index label)

In [ ]:
# Drop the column we just created — it was for demonstration
df_trimmed = df.drop(columns=["has_family"])
print("Columns remaining:", df_trimmed.columns.tolist())

In [ ]:
# You can also drop multiple columns at once
# This is common when you want to remove columns you know you won't use
df_analysis = df.drop(columns=["name"])
print("Shape after dropping name:", df_analysis.shape)

## Putting It Together: A First Look at the Titanic Data

Let's use what we've learned to ask a few quick, concrete questions about the dataset.

In [ ]:
print("=== TITANIC DATASET — FIRST LOOK ===")
print(f"Passengers: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
print()

print("Survival:")
print(df["survived"].value_counts().rename({0: "Did not survive", 1: "Survived"}))
print()

print("Passenger class:")
print(df["pclass"].value_counts().sort_index().rename({1: "1st class", 2: "2nd class", 3: "3rd class"}))
print()

print("Sex:")
print(df["sex"].value_counts())
print()

print("Age:")
print(df["age"].describe().round(1))

## Your Turn

1. `df.describe()` only shows numeric columns by default. Pass `include="all"` to see all columns. What additional information do you get about `name`, `sex`?

2. Select just the columns `pclass`, `sex`, `age`, and `survived`. Call `.describe()` on the result. Which of those four columns has the widest spread relative to its mean?

3. Add a column called `age_group` where passengers under 18 are `"child"`, those 18–60 are `"adult"`, and those over 60 are `"senior"`.
   *Hint: assign the column in three steps using boolean filters — `df.loc[df["age"] < 18, "age_group"] = "child"`, etc.*

4. After adding `age_group`, use `.value_counts()` on it. How many children were aboard?

In [ ]:
# Your code here